# T3.3 EIT-002 Environmental Footprint Simulation

**READ.ME**

This notebook calls the databases and project defined in EIT-003. The notebook is mainly a copy of EIT-001 as no example datasets for Simulation Configurations were shared yet.

- Setting the project based on EIT-003
- Read in Process Simulation Hub data
- Integration in foreground database as scenario 
- LCA results disaggregated in:
    - Total impact
    - Impact per tool / machining operation

### 1. Setting the project

In [1]:
import bw2data as bd
import bw2calc as bc
import bw2io as bi
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import bw2analyzer as bwa
from reed.utils import Config
import bw_processing as bwp
from bw2io.strategies.generic import set_code_by_activity_hash

08:37:35+0200 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.


In [2]:
config = Config()

In [3]:
bd.projects.set_current(config.PROJECT_NAME)

In [5]:
bonsai = bd.Database('BONSAI_V2.1.6')
bonsai_bio = bd.Database('BONSAI_V2.1.6 biosphere')
ecoinvent = bd.Database('ecoinvent-3.10-biosphere')
reed = bd.Database('reed')

#### 2. Import Simulation Data

This part draws from defined functions in the script, sensordata_analysis.py. It imports the monitoring data and calculates the energy consumption per tool and connect the tools to the reed tools

In [6]:
from reed.sensordata_analysis import (
    run_sensordata_analysis,
    find_missing_op_type_tools,
    compute_event_totals,
    split_final_df_by_company_event,
    sum_by_reed_tool,
    split_final_df_by_company_event,
)

In [7]:
simulation_data_path = config.REED_PATH/'Data'/"SimulationData"/"WWR"/"T2472"/"data"
tool_descriptions_path = config.REED_PATH/'Data'/"MonitoringData"

In [8]:
sensordata, event_summaries_df, ambiguous_company_tools, key_company_tools, key_reed_tools = run_sensordata_analysis(
    simulation_data_path,
    tool_descriptions_path,debug=True
)
print(sensordata)


Total CSV files imported: 4
                                          energy_consumption  Power_Active  \
company event_id             Tool_Number                                     
wwr     OF10004_G_TWB_7SZ421 0.0               199943.657180   6813.293128   
                             81.0               79448.274472   8209.350979   
                             82.0               23343.902749   8310.724871   
                             100.0              67116.527595   8319.371254   
                             102.0             118242.215355   9726.310414   
...                                                      ...           ...   
                             2694.0              4718.325664  13096.354965   
                             2695.0               959.320473   9700.993548   
                             2696.0              1756.356816  13036.875333   
                             2722.0               351.314922  10199.465472   
                             3350.0

Identify tools that have different operation type in the key_company_tools. These tool numbers are removed from the key to avoid an ambigious key. These tool numbers will be assigned to the generic machining process in the reed database

In [9]:
print(ambiguous_company_tools)

   company  Tool_Number                   op_type_candidates
0   goimek            2                 [AUXILIARY, MILLING]
1   goimek            6                 [AUXILIARY, MILLING]
2   goimek           30                 [AUXILIARY, MILLING]
3   goimek           31                   [CHAMFER, MILLING]
4   goimek           44                 [AUXILIARY, MILLING]
5   goimek           45                 [AUXILIARY, MILLING]
6  komatsu            3           [AUXILIARY, ROUGH MILLING]
7  komatsu           48  [AUXILIARY, MILLING, ROUGH MILLING]
8  komatsu           50        [ROUGH BORING, ROUGH MILLING]
9  komatsu           72      [FINISH MILLING, ROUGH MILLING]


In [10]:
#Identify missing op type tools, e.g. tools that are in the monitoring data but not in tool descriptions
missing_tool_descriptions = find_missing_op_type_tools(sensordata)

In [11]:
#Compute total time and energy per event per company
event_totals = compute_event_totals(sensordata, event_summaries_df)

### 2. Creating a scenario for Simulated Data integration

this section is missing a part about how to handle the processes that are not in hours?

In [12]:
from reed.swapping_data import (
    find_activities_by_name,
    build_scenarios_from_power_active,
    build_consumer_input_scenarios_from_hours,
    describe_scenarios,
    lcia_for_activity_with_scenario,
    find_subactivity_by_name_fragment,
    build_consumer_input_scenarios_from_hours_for_activity,
    get_product_id_for_activity,
)

Select which event you want to analyse

In [13]:
company = "wwr"
event_id = "OF10004_G_TWB_7SZ421"
product = "WWR total"
demand = 1

Start by selected the company and event_id where you want to swap in the sensordata

In [14]:
#Split sensordata by company and event
event_dfs = split_final_df_by_company_event(sensordata)

#Compute energy by reed tool for a specific company and event
tool_energy_by_company_and_event = sum_by_reed_tool(
    event_dfs[(company, event_id)]
)
print(tool_energy_by_company_and_event)

                                                          energy_consumption  \
company event_id             reed tool                                         
wwr     OF10004_G_TWB_7SZ421 boring, soraluce                   39487.347249   
                             drilling, soraluce                 54469.954977   
                             generic machining, soraluce       551951.801293   
                             milling, soraluce                 550725.038966   

                                                              hours  \
company event_id             reed tool                                
wwr     OF10004_G_TWB_7SZ421 boring, soraluce              3.190278   
                             drilling, soraluce            4.410833   
                             generic machining, soraluce  65.591389   
                             milling, soraluce            51.766944   

                                                          Power_Active  
company event_id   


Find all processes present in the sensordata in the reed-database

In [15]:
tools = tool_energy_by_company_and_event.index.get_level_values("reed tool").unique()
found = {t: find_activities_by_name(reed, t, exact=False) for t in tools}
found

{'boring, soraluce': ['boring, soraluce' (hour, GB, None),
  'boring, soraluce' (hour, ES, None),
  'boring, soraluce' (hour, DE, None)],
 'drilling, soraluce': ['drilling, soraluce' (kilogram, GB, None),
  'drilling, soraluce' (hour, DE, None),
  'drilling, soraluce' (hour, GB, None),
  'drilling, soraluce' (kilogram, DE, None),
  'drilling, soraluce' (kilogram, ES, None),
  'drilling, soraluce' (hour, ES, None)],
 'generic machining, soraluce': ['generic machining, soraluce' (kilogram, ES, None),
  'generic machining, soraluce' (kilogram, DE, None),
  'generic machining, soraluce' (hour, ES, None),
  'generic machining, soraluce' (hour, DE, None),
  'generic machining, soraluce' (kilogram, GB, None),
  'generic machining, soraluce' (hour, GB, None)],
 'milling, soraluce': ['milling, soraluce' (kilogram, DE, None),
  'milling, soraluce' (hour, ES, None),
  'milling, soraluce' (hour, GB, None),
  'milling, soraluce' (kilogram, GB, None),
  'milling, soraluce' (kilogram, ES, None),
  'm

Find subactivities called A3 for the product under study

In [16]:
product_activity = reed.get(name=product)

# Find the A3 activity inside KOMATSU total
a3_matches = find_subactivity_by_name_fragment(
    parent_activity=product_activity,
    name_fragment="A3",
)

print(a3_matches)
a3_activity = a3_matches[0]

['WWR A3: machining' (unit, DE, None)]


In [17]:
soraluce_matches = find_subactivity_by_name_fragment(
    parent_activity=a3_activity,
    name_fragment="soraluce machining",
)

print(soraluce_matches)

soraluce_activity = soraluce_matches[0]

['Soraluce machining, WWR' (unit, DE, None)]


In [18]:
#Identifying the electricity activity in bonsai to be able to compute the original electricity input amount for the activities found above
electricity = bonsai.get(name='production of electricity (market for)', location='DE', unit='TJ')

Create scenario where alle electricity inputs from the sensordata to the processes identified in found.

In [19]:
scenarios = build_scenarios_from_power_active(tool_energy_by_company_and_event, found, electricity)
scenarios


original  wwr_OF10004_G_TWB_7SZ421
product            activity                                              
318672587029032968 318722215686516736  0.000000                  0.044559
                   318722215216754688  0.000053                  0.044457
                   318722214990262272  0.000000                  0.030294
                   318722215191588864  0.000000                  0.038299

The input of products in found to consuming activities are found and used to build scenarios.

In [20]:
consumer_scenarios = build_consumer_input_scenarios_from_hours_for_activity(
    tool_energy_by_company_and_event,
    found,
    consumer_activity=soraluce_activity,
)
consumer_scenarios

,,original,wwr_OF10004_G_TWB_7SZ421
product,activity,,
318722215703293952,318722217511038976,3.190278,3.190278
318722215216754688,318722217511038976,4.410833,4.410833
318722214839267328,318722217511038976,65.591389,65.591389
318722215107702784,318722217511038976,51.766944,51.766944


In [21]:
full_scenario = pd.concat([scenarios, consumer_scenarios])

In [22]:
full_scenario_readable = describe_scenarios(full_scenario)
full_scenario_readable

,product_name,product_unit,activity_name,activity_unit,original,wwr_OF10004_G_TWB_7SZ421
0,production of electricity (market for),TJ,"boring, soraluce",hour,0.000000,0.044559
1,production of electricity (market for),TJ,"drilling, soraluce",hour,0.000053,0.044457
2,production of electricity (market for),TJ,"generic machining, soraluce",hour,0.000000,0.030294
3,production of electricity (market for),TJ,"milling, soraluce",hour,0.000000,0.038299
4,"boring, soraluce",hour,"Soraluce machining, WWR",unit,3.190278,3.190278
5,"drilling, soraluce",hour,"Soraluce machining, WWR",unit,4.410833,4.410833
6,"generic machining, soraluce",hour,"Soraluce machining, WWR",unit,65.591389,65.591389
7,"milling, soraluce",hour,"Soraluce machining, WWR",unit,51.766944,51.766944


#### 4. Initial LCA scores for Footprint Simulation

LCA score with sensor data

In [23]:
score, lca = lcia_for_activity_with_scenario(
    activity=product_activity,
    method=config.lcia_gwp100,
    scenario_df=full_scenario,
    scenario_col=None,     # "original" or None (for defined scenario)
    amount=demand
)
score


68454.30917342058

LCA score without sensor data

In [24]:
product_reed = product_activity
lca = product_reed.lca(method=config.lcia_gwp100)
lca.score

40993.59723243967

In [25]:
wwr_total = reed.get(name="WWR total", location="DE", unit="unit")

In [26]:
bwa.print_recursive_calculation(wwr_total,
lcia_method=config.lcia_gwp100,max_level=1,cutoff=0.001)

Fraction of score | Absolute score | Amount | Activity
0001 | 4.099e+04 |     1 | 'WWR total' (unit, DE, None)
  0.961 | 3.94e+04 |     1 | 'WWR A1: material inputs' (unit, DE, None)
  0.0127 | 518.9 |     1 | 'WWR A2: transport' (unit, DE, None)
  0.0571 |  2341 |     1 | 'WWR A3: machining' (unit, DE, None)
  -0.0309 | -1269 | 0.825 | 'treatment of steel, industrial waste (market for)' (ton, DE, None)


### 3. Results preparation

In [35]:
import importlib
import reed.LCA_functions
import json

importlib.reload(reed.LCA_functions)
from reed.LCA_functions import (get_machining_activity, simulation_footprint_output)

In [36]:
simulation_footprint_output_test = simulation_footprint_output(product_activity, lcia_method=config.lcia_gwp100, company=company, event_id=event_id)
print(json.dumps(simulation_footprint_output_test, indent=2))

Exported to simulation_wwr_OF10004_G_TWB_7SZ421_2026-06-08.json
{
  "product": "WWR total",
  "total_climate_impact_kg_co2eq": 40993.5972,
  "a1_raw_materials": {
    "total_kg_co2eq": 39402.0837
  },
  "a2_transport": {
    "total_kg_co2eq": 518.9271
  },
  "a3_machining": {
    "total_kg_co2eq": 124.9595,
    "operations": [
      {
        "operation": "generic machining, soraluce",
        "amount": 65.591389
      },
      {
        "operation": "boring, soraluce",
        "amount": 3.190278
      },
      {
        "operation": "drilling, soraluce",
        "amount": 4.410833
      },
      {
        "operation": "milling, soraluce",
        "amount": 51.766944
      },
      {
        "operation": "soraluce machine, WWR",
        "amount": 9.1e-05
      }
    ]
  }
}
